# Misinformation and False Memories RAG System

This project develops a Retrieval-Augmented Generation (RAG) system for answering research questions about **how post-event information can alter human memory**. The knowledge base consists of **58 academic PDF papers** covering the misinformation effect, eyewitness and social memory, source monitoring, digital misinformation, suggestion, correction, warnings, repetition, individual susceptibility, and related cognitive and neural mechanisms.

The documents are stored in `data/raw/` and will be transformed through text extraction, cleaning, chunking, embedding, vector storage, semantic retrieval, and LLM-based generation. The final system will produce answers grounded in the retrieved research sources and will be evaluated for retrieval quality, answer relevance, and faithfulness to the corpus.

### Imports and Dependencies

This section imports the libraries required for document processing, data analysis, embedding generation, vector storage, retrieval, and RAG development. Imports are kept separate from configuration and data processing to make the notebook's dependencies clear and reproducible.

In [1]:
!pip install PyMuPDF


Defaulting to user installation because normal site-packages is not writeable


In [2]:
from pathlib import Path
from collections import Counter
from datetime import date

import re
import os

import fitz
import pandas as pd

## Task 1: Document Collection and Logging

### Identify the Collected Documents

This step searches the raw data directory for all supported source documents. The resulting file list becomes the input to the dataset statistics, collection log, text extraction, and later preprocessing stages.

The current corpus is expected to contain 58 retained research papers.

In [3]:
RAW_DATA_DIR = Path("data/raw")

pdf_files = sorted(RAW_DATA_DIR.glob("*.pdf"))
txt_files = sorted(RAW_DATA_DIR.glob("*.txt"))
all_files = pdf_files + txt_files

if not all_files:
    raise ValueError("No PDF or TXT documents found in data/raw.")

print(f"Documents found: {len(all_files)}")

Documents found: 58


### Calculate Dataset Statistics

This step measures the size of the collected corpus before any cleaning or transformation takes place.

For each document, the notebook records its file size, number of pages, and extracted word count. These values are then aggregated to report the total number of documents, pages, words, and storage size of the dataset. 

These statistics provide a baseline for understanding the scale of the RAG knowledge base.

In [4]:
total_pages = total_words = total_size = 0

for filepath in all_files:
    total_size += filepath.stat().st_size

    if filepath.suffix.lower() == ".pdf":
        with fitz.open(filepath) as doc:
            total_pages += len(doc)
            total_words += sum(len(page.get_text().split()) for page in doc)
    else:
        total_words += len(filepath.read_text(encoding="utf-8", errors="ignore").split())

print("DATASET STATISTICS")
print("-" * 30)
print(f"Documents : {len(all_files):,}")
print(f"PDF       : {len(pdf_files):,}")
print(f"TXT       : {len(txt_files):,}")
print(f"Pages     : {total_pages:,}")
print(f"Words     : {total_words:,}")
print(f"Size      : {total_size / (1024**2):.2f} MB")

DATASET STATISTICS
------------------------------
Documents : 58
PDF       : 58
TXT       : 0
Pages     : 1,055
Words     : 629,043
Size      : 45.01 MB


### Build the Collection Log

The collection log records basic information about every source in the corpus. This creates a traceable record of the documents used in the
RAG system and provides metadata that can be used later for attribution, filtering, and evaluation.

The log records the filename, source URL, access date, file type, file size, and available licence or usage information.

In [5]:
collection_log = pd.DataFrame({
    "filename": [f.name for f in all_files],
    "type": [f.suffix.lower().replace(".", "").upper() for f in all_files],
    "size_mb": [f.stat().st_size / (1024**2) for f in all_files],
    "date_accessed": date.today().isoformat()
})

collection_log.head()

,filename,type,size_mb,date_accessed
0,10.1177_2167702617724424.pdf,PDF,0.979162,2026-10-01
1,13415_2024_Article_1183.pdf,PDF,2.017839,2026-10-01
2,13421_2022_Article_1328.pdf,PDF,1.162881,2026-10-01
3,1744-9081-8-35.pdf,PDF,0.263000,2026-10-01
4,41235_2020_Article_262.pdf,PDF,1.192057,2026-10-01


### Record Source URLs and Access Dates

Source URLs are recorded to preserve the origin of each research paper and make the corpus traceable. When a DOI is available in the document,
it is converted into a standard DOI URL.

The access date records when the document was collected for this project. If a source URL cannot be identified automatically, it is
marked clearly rather than being guessed.

In [6]:
def extract_url(filepath):
    if filepath.suffix.lower() != ".pdf":
        return "Source URL not identified"

    with fitz.open(filepath) as doc:
        text = "\n".join(page.get_text() for page in doc[:2])

    doi = re.search(r"(10\.\d{4,9}/[-._;()/:A-Z0-9]+)", text, re.I)
    return f"https://doi.org/{doi.group(1).rstrip('.,')}" if doi else "Source URL not identified"

collection_log["source_url"] = [
    extract_url(f) for f in all_files
]
collection_log["date_accessed"] = date.today().isoformat()

collection_log.head()

,filename,type,size_mb,date_accessed,source_url
0,10.1177_2167702617724424.pdf,PDF,0.979162,2026-10-01,https://doi.org/10.1177/2167702617724424
1,13415_2024_Article_1183.pdf,PDF,2.017839,2026-10-01,https://doi.org/10.3758/s13415-024-01183-y
2,13421_2022_Article_1328.pdf,PDF,1.162881,2026-10-01,https://doi.org/10.3758/s13421-022-01328-9
3,1744-9081-8-35.pdf,PDF,0.263000,2026-10-01,Source URL not identified
4,41235_2020_Article_262.pdf,PDF,1.192057,2026-10-01,https://doi.org/10.1186/s41235-020-00262-1


### Record Licence and Terms of Use

Licence information is recorded where it can be identified from the document. Explicit Creative Commons licences are labelled using the
licence stated in the paper.

When no explicit licence is identified, the document is treated as subject to the publisher's copyright and terms rather than assuming
that it is freely reusable. This distinction helps maintain appropriate source attribution and transparency for the academic RAG project.

In [7]:
def detect_license(filepath):
    if filepath.suffix.lower() != ".pdf":
        return "Not identified"

    with fitz.open(filepath) as doc:
        text = "\n".join(page.get_text() for page in doc[:2]).lower()

    if "creative commons attribution 4.0" in text or "cc by 4.0" in text:
        return "CC BY 4.0"
    if "creative commons" in text:
        return "Creative Commons licence stated"
    if "open access" in text:
        return "Open Access; licence not identified"

    return "Publisher copyright/terms; licence not identified"

collection_log["licence_or_terms"] = [
    detect_license(f) for f in all_files
]

### Verify Document Readability

Before moving to preprocessing, a small sample of documents is checked to confirm that text can be extracted successfully.

This helps identify scanned, corrupted, or otherwise unreadable files before they enter the cleaning and chunking stages. The check displays
the first 500 characters from three different documents.

In [8]:
for filepath in pdf_files[:3]:
    with fitz.open(filepath) as doc:
        text = " ".join(page.get_text() for page in doc)

    print(f"\n{filepath.name}")
    print("-" * 60)
    print(text[:500].strip())


10.1177_2167702617724424.pdf
------------------------------------------------------------
https://doi.org/10.1177/2167702617724424
Clinical Psychological Science
2017, Vol. 5(6) 1048­–1069
© The Author(s) 2017
Reprints and permissions: 
sagepub.com/journalsPermissions.nav
DOI: 10.1177/2167702617724424
www.psychologicalscience.org/CPS
Theoretical/Methodological/Review Article
Memory aberrations are notable characteristics of post-
traumatic stress disorder (PTSD) and depression. For 
example, individuals with these disorders tend to recol-
lect emotionally negative experiences better

13415_2024_Article_1183.pdf
------------------------------------------------------------
Vol:.(1234567890)
Cognitive, Affective, & Behavioral Neuroscience (2024) 24:440–452
https://doi.org/10.3758/s13415-024-01183-y
RESEARCH ARTICLE
Warning before misinformation exposure modulates memory 
encoding
Jessica M. Karanian1   · Ayanna K. Thomas2   · Elizabeth Race2 
Accepted: 1 March 2024 / Published online: 19

### Task 1 Summary

The document collection has been identified, measured, and logged. The corpus statistics establish the size of the dataset, while the
collection log preserves source, access, file, and licence information.

A sample readability check also confirms that the collected PDFs contain extractable text. The completed collection log is saved for use in later
stages of the project.

In [9]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

collection_log.to_csv(
    OUTPUT_DIR / "collection_log.csv",
    index=False
)

print("Collection log saved to outputs/collection_log.csv")
print(f"Logged documents: {len(collection_log)}")

Collection log saved to outputs/collection_log.csv
Logged documents: 58


## Task 2: Document Cleaning

This stage cleans the text extracted from the collected research papers before it is chunked and embedded. The goal is to remove extraction
noise while preserving information that may be useful for retrieval.

The cleaning pipeline applies Unicode normalization, repeated header/footer removal, whitespace normalization, and duplicate paragraph
removal. Cleaned documents are saved in `data/clean` so that the raw source files remain unchanged.

Headings and paragraph boundaries are deliberately preserved because they provide useful structural information for later chunking and
retrieval.

### Imports for Cleaning

This section imports the libraries required to clean the extracted research text. The cleaning process uses Unicode normalization, regular expressions, and tabular statistics to identify and measure changes made to the corpus.

In [10]:
import re
import unicodedata

### Define the Cleaning Pipeline

The cleaning pipeline applies four techniques to the extracted text: Unicode normalization, whitespace normalization, repeated line removal, and duplicate paragraph removal.

The cleaned documents are stored separately in `data/clean`, ensuring that the original source files in `data/raw` remain unchanged.

In [11]:
CLEAN_DIR = Path("data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)


def normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)


def normalize_whitespace(text):
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def remove_repeated_lines(text):
    lines = text.splitlines()
    counts = pd.Series(lines).value_counts()

    repeated = {
        line for line, count in counts.items()
        if count >= 3 and 20 <= len(line) <= 150
    }

    return "\n".join(line for line in lines if line not in repeated)


def remove_duplicate_paragraphs(text):
    paragraphs = re.split(r"\n\s*\n", text)
    seen = set()
    unique = []

    for paragraph in paragraphs:
        key = re.sub(r"\s+", " ", paragraph).strip().lower()

        if key and key not in seen:
            seen.add(key)
            unique.append(paragraph.strip())

    return "\n\n".join(unique)


def clean_text(text):
    text = normalize_unicode(text)
    text = remove_repeated_lines(text)
    text = normalize_whitespace(text)
    return remove_duplicate_paragraphs(text)

### Clean and Save the Corpus

Each PDF is read and passed through the cleaning pipeline. The original and cleaned word counts are recorded so that the effect of preprocessing
can be measured across the corpus.

The cleaned text is saved as `.txt` files in `data/clean`.

In [12]:
cleaning_stats = []

for filepath in pdf_files:
    with fitz.open(filepath) as doc:
        raw_text = "\n".join(page.get_text() for page in doc)

    cleaned_text = clean_text(raw_text)

    (CLEAN_DIR / f"{filepath.stem}.txt").write_text(
        cleaned_text,
        encoding="utf-8"
    )

    before = len(raw_text.split())
    after = len(cleaned_text.split())

    cleaning_stats.append({
        "filename": filepath.name,
        "words_before": before,
        "words_after": after,
        "words_removed": before - after
    })

cleaning_stats = pd.DataFrame(cleaning_stats)

print(f"Documents cleaned: {len(cleaning_stats)}")
print(f"Output directory: {CLEAN_DIR}")

Documents cleaned: 58
Output directory: data/clean


## Before and After Samples

Three documents are inspected before and after cleaning to demonstrate the effect of the preprocessing steps.

The samples help verify that extraction noise has been reduced while the main research content remains intact.

In [13]:
for filepath in pdf_files[:3]:
    with fitz.open(filepath) as doc:
        before = "\n".join(page.get_text() for page in doc)

    after = (CLEAN_DIR / f"{filepath.stem}.txt").read_text(
        encoding="utf-8"
    )

    print(f"\n{'=' * 70}")
    print(filepath.name)
    print(f"{'=' * 70}")

    print("\nBEFORE:")
    print(before[:500])

    print("\nAFTER:")
    print(after[:500])



10.1177_2167702617724424.pdf

BEFORE:
https://doi.org/10.1177/2167702617724424
Clinical Psychological Science
2017, Vol. 5(6) 1048­–1069
© The Author(s) 2017
Reprints and permissions: 
sagepub.com/journalsPermissions.nav
DOI: 10.1177/2167702617724424
www.psychologicalscience.org/CPS
Theoretical/Methodological/Review Article
Memory aberrations are notable characteristics of post-
traumatic stress disorder (PTSD) and depression. For 
example, individuals with these disorders tend to recol-
lect emotionally negative experiences better 

AFTER:
https://doi.org/10.1177/2167702617724424
Clinical Psychological Science
2017, Vol. 5(6) 1048­–1069
© The Author(s) 2017
Reprints and permissions: 
sagepub.com/journalsPermissions.nav
DOI: 10.1177/2167702617724424
www.psychologicalscience.org/CPS
Theoretical/Methodological/Review Article
Memory aberrations are notable characteristics of post-
traumatic stress disorder (PTSD) and depression. For 
example, individuals with these disorders tend to reco

### Cleaning Statistics

This step reports the aggregate effect of cleaning across the entire corpus.

The statistics show the total number of words before and after cleaning and the number of words removed during preprocessing.

In [14]:
total_before = cleaning_stats["words_before"].sum()
total_after = cleaning_stats["words_after"].sum()

print("CLEANING STATISTICS")
print("-" * 30)
print(f"Documents       : {len(cleaning_stats):,}")
print(f"Words before    : {total_before:,}")
print(f"Words after     : {total_after:,}")
print(f"Words removed   : {total_before - total_after:,}")

CLEANING STATISTICS
------------------------------
Documents       : 58
Words before    : 629,043
Words after     : 615,234
Words removed   : 13,809


### Count Duplicate Paragraphs

Duplicate paragraphs are measured separately from the overall word reduction because words can also be removed through whitespace and
repeated-line cleaning.

This gives a more accurate record of the deduplication performed on the corpus.

In [15]:
def count_duplicate_paragraphs(text):
    paragraphs = re.split(r"\n\s*\n", text)

    normalized = [
        re.sub(r"\s+", " ", paragraph).strip().lower()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    return len(normalized) - len(set(normalized))


duplicates_dropped = 0

for filepath in pdf_files:
    with fitz.open(filepath) as doc:
        text = "\n".join(page.get_text() for page in doc)

    duplicates_dropped += count_duplicate_paragraphs(text)

print(f"Duplicate paragraphs dropped: {duplicates_dropped:,}")

Duplicate paragraphs dropped: 0


## Task 3: Extraction, Chunking & Metadata

## Imports for Extraction and Chunking

This section imports the tools required for sentence-based chunking and saving the final chunk dataset. The extraction itself uses the cleaned text files produced in Task 2.

In [16]:
!pip install nltk

Defaulting to user installation because normal site-packages is not writeable


In [17]:
import json
import nltk 
from nltk.tokenize import sent_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

/home/Student/.local/lib/python3.10/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/Student/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


True

### Extract and Validate Text

The cleaned text files are loaded and checked for empty or failed extractions. A log is maintained so that every source can be traced through the preprocessing pipeline.

Only successfully extracted documents are passed to chunking.

In [18]:
clean_files = sorted(CLEAN_DIR.glob("*.txt"))

documents = {}
extraction_log = []

for filepath in clean_files:
    try:
        text = filepath.read_text(encoding="utf-8", errors="ignore").strip()
        status = "ok" if text else "empty"

        if text:
            documents[filepath.name] = text

    except Exception as e:
        text, status = "", f"failed: {type(e).__name__}"

    extraction_log.append({
        "filename": filepath.name,
        "words": len(text.split()),
        "status": status
    })

extraction_log = pd.DataFrame(extraction_log)

print(f"Documents processed: {len(clean_files)}")
print(f"Successful: {len(documents)}")
print(f"Failed/empty: {(extraction_log['status'] != 'ok').sum()}")

extraction_log[extraction_log["status"] != "ok"]

Documents processed: 58
Successful: 58
Failed/empty: 0


,filename,words,status


### Compare Two Chunking Strategies

Two chunking strategies are applied to the same cleaned corpus: recursive character chunking and sentence-based chunking.

Both use a target size of 1,000 characters. Their number of chunks and average chunk length are compared, while example outputs show how each strategy preserves the structure of the research text.

In [19]:
CHUNK_SIZE = 1000
RECURSIVE_OVERLAP = 150
SENTENCE_OVERLAP = 2


def recursive_chunks(text, size=1000, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks


def sentence_chunks(text, size=1000, overlap=2):
    sentences = sent_tokenize(text)
    chunks = []
    current = []

    for sentence in sentences:
        candidate = " ".join(current + [sentence])

        if len(candidate) <= size:
            current.append(sentence)
        else:
            if current:
                chunks.append(" ".join(current))

            current = current[-overlap:] + [sentence]

    if current:
        chunks.append(" ".join(current))

    return chunks


recursive = [
    chunk
    for text in documents.values()
    for chunk in recursive_chunks(text, CHUNK_SIZE, RECURSIVE_OVERLAP)
]

sentence = [
    chunk
    for text in documents.values()
    for chunk in sentence_chunks(text, CHUNK_SIZE, SENTENCE_OVERLAP)
]


def chunk_metrics(chunks):
    lengths = [len(chunk) for chunk in chunks]

    return {
        "chunks": len(chunks),
        "avg_characters": round(sum(lengths) / len(lengths)),
        "avg_words": round(
            sum(len(chunk.split()) for chunk in chunks) / len(chunks)
        )
    }


comparison = pd.DataFrame({
    "Recursive Character": chunk_metrics(recursive),
    "Sentence Based": chunk_metrics(sentence)
}).T

print("CHUNKING COMPARISON")
print(comparison)

print("\nRECURSIVE CHARACTER EXAMPLE")
print(recursive[0][:500])

print("\nSENTENCE-BASED EXAMPLE")
print(sentence[0][:500])

CHUNKING COMPARISON
                     chunks  avg_characters  avg_words
Recursive Character    4889             994        149
Sentence Based         7531             924        138

RECURSIVE CHARACTER EXAMPLE
https://doi.org/10.1177/2167702617724424
Clinical Psychological Science
2017, Vol. 5(6) 1048­–1069
© The Author(s) 2017
Reprints and permissions: 
sagepub.com/journalsPermissions.nav
DOI: 10.1177/2167702617724424
www.psychologicalscience.org/CPS
Theoretical/Methodological/Review Article
Memory aberrations are notable characteristics of post-
traumatic stress disorder (PTSD) and depression. For 
example, individuals with these disorders tend to recol-
lect emotionally negative experiences better 

SENTENCE-BASED EXAMPLE
https://doi.org/10.1177/2167702617724424
Clinical Psychological Science
2017, Vol. 5(6) 1048­–1069
© The Author(s) 2017
Reprints and permissions: 
sagepub.com/journalsPermissions.nav
DOI: 10.1177/2167702617724424
www.psychologicalscience.org/CPS
Theoretical/Met

### Select the Final Strategy and Create Metadata

The sentence-based strategy is selected because it preserves sentence boundaries, which is important for academic text and helps keep related ideas together.

A chunk size of 1,000 characters and an overlap of 2 sentences are used. Each chunk is assigned metadata containing the source filename, a unique chunk ID, and its sequential position within the source document.

In [20]:
FINAL_SIZE = 1000
FINAL_OVERLAP = 2

final_chunks = []

for filename, text in documents.items():
    chunks = sentence_chunks(
        text,
        size=FINAL_SIZE,
        overlap=FINAL_OVERLAP
    )

    for position, chunk in enumerate(chunks):
        final_chunks.append({
            "text": chunk,
            "metadata": {
                "filename": filename,
                "chunk_id": f"{Path(filename).stem}_{position:04d}",
                "position": position
            }
        })

print(f"Final chunks created: {len(final_chunks):,}")

Final chunks created: 7,531


### Inspect and Save Final Chunks

The first five chunks are inspected to verify that the text and metadata were created correctly. The complete chunk dataset is then saved as JSON
for use in the embedding and vector-storage stages.

In [21]:
for i, item in enumerate(final_chunks[:5], 1):
    print(f"\n{'=' * 70}")
    print(f"CHUNK {i}")
    print("Metadata:", item["metadata"])
    print("Text:", item["text"][:500])

Path("data/processed").mkdir(parents=True, exist_ok=True)

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(final_chunks, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(final_chunks):,} chunks to data/processed/chunks.json")


CHUNK 1
Metadata: {'filename': '10.1177_2167702617724424.txt', 'chunk_id': '10.1177_2167702617724424_0000', 'position': 0}
Text: https://doi.org/10.1177/2167702617724424
Clinical Psychological Science
2017, Vol. 5(6) 1048­–1069
© The Author(s) 2017
Reprints and permissions: 
sagepub.com/journalsPermissions.nav
DOI: 10.1177/2167702617724424
www.psychologicalscience.org/CPS
Theoretical/Methodological/Review Article
Memory aberrations are notable characteristics of post-
traumatic stress disorder (PTSD) and depression. For 
example, individuals with these disorders tend to recol-
lect emotionally negative experiences better 

CHUNK 2
Metadata: {'filename': '10.1177_2167702617724424.txt', 'chunk_id': '10.1177_2167702617724424_0001', 'position': 1}
Text: It is not surprising that many studies have 
focused on the links among PTSD, depression, and 
memory. The bulk of this research has been concerned 
with the amount of information that can be correctly 
remembered (Koriat, Goldsmith, & Pan

## Task 4: Embedding Generation

### Choose and Load the Embedding Model

The embedding step uses Google's Gemini Embedding API to convert each
text chunk into a numerical vector. NumPy is used to store and work
with the resulting vectors, while `time` is used to measure embedding
execution time.

In [22]:
from google import genai
from google.genai import types

import numpy as np
import time
import os

### Configure the Gemini API

The Gemini API client is initialized using an API key stored in an
environment variable. The key is not written directly in the notebook
to avoid exposing credentials.

The embedding model used is `gemini-embedding-001`.

In [23]:
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

EMBEDDING_MODEL = "gemini-embedding-001"

print("Embedding model:", EMBEDDING_MODEL)

Embedding model: gemini-embedding-001


### Choose the Embedding Model

This project uses Google's `gemini-embedding-001` model to generate embeddings for the academic corpus.

The model supports up to 2,048 input tokens and can produce embeddings with different dimensionalities. A dimensionality of 768 is selected for this project to provide a substantial vector representation while
keeping vector storage smaller than the default 3,072 dimensions.

The `RETRIEVAL_DOCUMENT` task type is used because the inputs are research passages that will later be retrieved in response to queries.

In [24]:
OUTPUT_DIMENSION = 768
MAX_TOKENS = 2048

print("Model:", EMBEDDING_MODEL)
print("Output dimensions:", OUTPUT_DIMENSION)
print("Maximum input tokens:", MAX_TOKENS)

Model: gemini-embedding-001
Output dimensions: 768
Maximum input tokens: 2048


### Verify Chunk Lengths

Before generating embeddings, the chunks are checked against the
embedding model's maximum input length of 2,048 tokens.

The chunks created in Task 3 were limited by character length, so this
step verifies their actual token lengths before sending them to the API.

In [25]:
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)

print("Client recreated successfully.")

Enter your Gemini API key:  ········


Client recreated successfully.


In [27]:
MAX_TOKENS = 2048
CHUNK_SIZE = 1000

chunk_lengths = [
    len(item["text"])
    for item in final_chunks
]

estimated_tokens = [
    length / 4
    for length in chunk_lengths
]

print("Maximum model tokens :", MAX_TOKENS)
print("Maximum chunk chars  :", max(chunk_lengths))
print(f"Estimated max tokens : {max(estimated_tokens):.0f}")
print(
    "Chunks exceeding 2,048 estimated tokens:",
    sum(tokens > MAX_TOKENS for tokens in estimated_tokens)
)

Maximum model tokens : 2048
Maximum chunk chars  : 3963
Estimated max tokens : 991
Chunks exceeding 2,048 estimated tokens: 0


### Handle Oversized Chunks

No final chunks exceed the embedding model's input limit, so the original
sentence-based chunks are retained without additional splitting.

In [30]:
embedding_chunks = final_chunks

print("Chunks ready for embedding:", len(embedding_chunks))

Chunks ready for embedding: 7531


### Generate Embeddings for All Chunks


The corpus is embedded in small batches with a delay between requests to
remain within the Gemini API free-tier request limit. Progress and total
execution time are recorded.

In [ ]:
texts = [item["text"] for item in embedding_chunks]

BATCH_SIZE = 20
DELAY = 12

embeddings = []
start_time = time.time()

for start in range(0, len(texts), BATCH_SIZE):
    batch = texts[start:start + BATCH_SIZE]

    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=batch,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT",
            output_dimensionality=OUTPUT_DIMENSION
        )
    )

    embeddings.extend(
        [embedding.values for embedding in result.embeddings]
    )

    completed = min(start + BATCH_SIZE, len(texts))
    print(f"Embedded {completed:,}/{len(texts):,}")

    if completed < len(texts):
        time.sleep(DELAY)

elapsed = time.time() - start_time
embeddings = np.array(embeddings)

print("\nEMBEDDING GENERATION COMPLETE")
print("-" * 35)
print(f"Chunks embedded : {len(embeddings):,}")
print(f"Dimensions      : {embeddings.shape[1]}")
print(f"Execution time  : {elapsed:.2f} seconds")

Embedded 20/7,531
Embedded 40/7,531
Embedded 60/7,531
Embedded 80/7,531
Embedded 100/7,531
Embedded 120/7,531


### Demonstrate Vector Similarity

Cosine similarity measures how closely two embedding vectors point in the
same direction. Higher values indicate greater semantic similarity, while
lower values indicate weaker semantic similarity.

A high-similarity pair and a low-similarity pair are identified from the
corpus and their corresponding text is displayed for inspection.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)


np.fill_diagonal(similarity_matrix, -1)


high_pair = np.unravel_index(
    np.argmax(similarity_matrix),
    similarity_matrix.shape
)


low_pair = np.unravel_index(
    np.argmin(similarity_matrix),
    similarity_matrix.shape
)

def show_pair(pair, label):
    i, j = pair

    print(f"\n{'=' * 70}")
    print(label)
    print(f"Cosine similarity: {similarity_matrix[i, j]:.4f}")
    print("=" * 70)

    print("\nCHUNK 1")
    print(embedding_chunks[i]["metadata"])
    print(embedding_chunks[i]["text"][:600])

    print("\nCHUNK 2")
    print(embedding_chunks[j]["metadata"])
    print(embedding_chunks[j]["text"][:600])


show_pair(high_pair, "HIGH-SIMILARITY PAIR")
show_pair(low_pair, "LOW-SIMILARITY PAIR")

### Interpretation

The high-similarity pair should contain text discussing closely related
concepts within misinformation and false-memory research. Its higher cosine
similarity demonstrates that the embedding model places semantically related
content closer together in vector space.

The low-similarity pair should discuss less-related concepts. Its lower cosine
similarity demonstrates that the embeddings can distinguish between different
semantic content.

This provides an initial validation that the generated embeddings are
suitable for semantic retrieval in the RAG system.

## Task 5: Vector Storage & Retrieval Setup

### Imports for Vector Storage

ChromaDB is used as the local vector database. The database will store
the chunk text, pre-generated BGE embeddings, and chunk metadata.

In [ ]:
import chromadb

### Create the Persistent Vector Database

The vector database is stored in `data/vectorstore/`, allowing the embeddings to remain available between notebook sessions.

The collection uses cosine distance because the embeddings were normalized during Task 4.

In [ ]:
VECTOR_DB_DIR = "data/vectorstore"

client = chromadb.PersistentClient(path=VECTOR_DB_DIR)

collection = client.get_or_create_collection(
    name="misinformation_memory",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

print("Vector database:", VECTOR_DB_DIR)
print("Collection:", collection.name)
print("Distance metric: cosine")

### Store Chunks, Embeddings and Metadata

Each processed chunk is stored together with its pre-generated embedding and metadata from Task 3.

The embeddings are not regenerated by Chroma; the vectors produced by `BAAI/bge-small-en-v1.5` are inserted directly into the collection.

In [ ]:
ids = [
    item["metadata"]["chunk_id"]
    for item in processed_chunks
]

texts = [
    item["text"]
    for item in processed_chunks
]

metadatas = [
    item["metadata"]
    for item in processed_chunks
]

vectors = embeddings.tolist()

batch_size = 5000

for start in range(0, len(ids), batch_size):
    end = start + batch_size

    collection.add(
        ids=ids[start:end],
        documents=texts[start:end],
        metadatas=metadatas[start:end],
        embeddings=vectors[start:end]
    )

print(f"Stored chunks: {collection.count():,}")

### Verify Persistence and Reloading

A new Chroma client is created using the same directory. The existing
collection is then loaded from disk.

No embedding model encoding is performed in this step. This demonstrates
that the stored vectors persist independently of the embedding-generation
step.

In [ ]:
reloaded_client = chromadb.PersistentClient(
    path=VECTOR_DB_DIR
)

reloaded_collection = reloaded_client.get_collection(
    name="misinformation_memory"
)

print("Reloaded collection:", reloaded_collection.name)
print("Stored chunks:", reloaded_collection.count())

### Implement Retrieval

The `retrieve()` function converts a user question into a BGE query embedding and searches the persistent Chroma collection.

It returns the top-k chunks together with their similarity scores and metadata.

Chroma returns cosine distance, where smaller values indicate closer vectors. The function converts this to cosine similarity using
`1 - distance`, so higher scores indicate more similar results.

In [ ]:
QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
)

def retrieve(query, k=5):
    query_embedding = embedding_model.encode(
        [QUERY_INSTRUCTION + query],
        normalize_embeddings=True
    )

    results = reloaded_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved.append({
            "text": document,
            "score": 1 - distance,
            "metadata": metadata
        })

    return retrieved

### Retrieval Test 1

The first question tests retrieval of research about the core
misinformation effect and how post-event information changes memory.

In [ ]:
query_1 = (
    "How does post-event misinformation affect people's memory "
    "for an original event?"
)

results_1 = retrieve(query_1, k=5)

for i, result in enumerate(results_1, 1):
    print(f"\nRESULT {i}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['metadata']['filename']}")
    print(f"Chunk: {result['metadata']['chunk_id']}")
    print(result["text"][:700])

### Retrieval Test 2
 
The second question tests whether retrieval can identify research concerning factors that make individuals more susceptible to
misinformation-induced memory distortion.

In [ ]:
query_2 = (
    "What factors make some people more susceptible to the "
    "misinformation effect than others?"
)

results_2 = retrieve(query_2, k=5)

for i, result in enumerate(results_2, 1):
    print(f"\nRESULT {i}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['metadata']['filename']}")
    print(f"Chunk: {result['metadata']['chunk_id']}")
    print(result["text"][:700])

### Retrieval Test 3

The third question tests retrieval of research about whether warnings or corrections can reduce the effects of misleading post-event
information.

In [ ]:
query_3 = (
    "Can warnings or corrections reduce memory distortion caused "
    "by misleading post-event information?"
)

results_3 = retrieve(query_3, k=5)

for i, result in enumerate(results_3, 1):
    print(f"\nRESULT {i}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['metadata']['filename']}")
    print(f"Chunk: {result['metadata']['chunk_id']}")
    print(result["text"][:700])

### Retrieval Inspection

The retrieved results are summarized by query, rank, similarity score, source filename, and chunk identifier. The retrieved text is included
for manual relevance checking before LLM integration.

In [ ]:
queries = {
    "Q1": (query_1, results_1),
    "Q2": (query_2, results_2),
    "Q3": (query_3, results_3)
}

retrieval_test = []

for query_id, (query, results) in queries.items():
    for rank, result in enumerate(results, 1):
        retrieval_test.append({
            "query": query_id,
            "rank": rank,
            "score": round(result["score"], 4),
            "filename": result["metadata"]["filename"],
            "chunk_id": result["metadata"]["chunk_id"],
            "text": result["text"][:300]
        })

retrieval_test = pd.DataFrame(retrieval_test)

retrieval_test[
    ["query", "rank", "score", "filename", "chunk_id", "text"]
]